In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))


# Fase 5.1 — Diagnóstico de probabilidades y umbrales

Se carga el Logit 5.0 congelado de nueve predictores y las predicciones guardadas. No se ajusta ni recalibra el modelo, ni se altera WoE o coeficientes.

**Límite de la validación interna:** el modelo ya se ajustó con todo TRAIN. Separar ahora un 30% estratificado permite un diagnóstico reproducible de umbrales usando solo TRAIN, pero **no crea una validación independiente del ajuste**. La validación fuera de muestra de umbrales requeriría un conjunto no usado al ajustar o un procedimiento de ajuste interno, fuera del alcance de esta fase sin reentrenamiento.

Los candidatos se congelan antes de calcular las tablas TEST/OOT. F1 máximo, recall mínimo del 50% y capacidad máxima del 10% son **escenarios ilustrativos configurables**, no objetivos de negocio acordados ni una recomendación de umbral óptimo. Los deciles son descriptivos por muestra y no modifican umbrales.

## UMB.1 — Importación y configuración

**Recibe:** Módulo de diagnóstico 5.1.

**Produce:** umb, CONFIG y carpeta independiente.

**Funciones:** 5.1.0.evaluacion_umbrales.py: config, threshold_grid.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: config, threshold_grid
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
spec = importlib.util.spec_from_file_location('evaluacion_umbrales', '5.1.0.evaluacion_umbrales.py')
umb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(umb)
CONFIG = umb.config()
EVAL_ROOT = Path('5.1.EVALUACION_UMBRALES')
for folder in ['reportes','graficos','evidencias']:
    (EVAL_ROOT / folder).mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_rows', 160)
pd.set_option('display.max_columns', 30)
display(pd.Series(CONFIG, name='Valor').to_frame())
print(umb.LIMITATION)
print('Grilla:', umb.threshold_grid()[0], 'a', umb.threshold_grid()[-1], 'paso 0.01; regla p >= umbral.')
CONFIG = umb.config(_central["threshold_diagnostics"])


## UMB.2 — Carga del modelo y predicciones congeladas

**Recibe:** Modelo, coeficientes, matrices, targets y predicciones 5.0.

**Produce:** congelado, modelo_congelado y predicciones_originales.

**Funciones:** 5.1.0.evaluacion_umbrales.py: load_frozen.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: load_frozen
congelado = umb.load_frozen()
modelo_congelado = congelado['model']
predicciones_originales = congelado['data']
coeficientes_iniciales = modelo_congelado.params.copy(deep=True)
display(congelado['validation'])
display(pd.DataFrame({'Predictor congelado': congelado['features']}))
print('Probabilidades originales reproducidas por el modelo cargado; sin fit ni transformaciones nuevas.')

## UMB.3 — Distribución de probabilidades

**Recibe:** Predicciones TRAIN/TEST/OOT existentes.

**Produce:** df_distribucion_probabilidades y figura.

**Funciones:** 5.1.0.evaluacion_umbrales.py: probability_summary, plot_distribution.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: probability_summary, plot_distribution
df_distribucion_probabilidades = umb.probability_summary(predicciones_originales)
display(df_distribucion_probabilidades)
figuras_umbrales = {}
figuras_umbrales['distribucion_probabilidades'] = umb.plot_distribution(predicciones_originales)
plt.show()

## UMB.4 — Partición interna de TRAIN para umbrales

**Recibe:** Solo predicciones y targets TRAIN; seed/fraction configurados.

**Produce:** train_validacion_interna, train_resto y manifiesto de partición.

**Funciones:** 5.1.0.evaluacion_umbrales.py: internal_train_partition.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: internal_train_partition
train_validacion_interna, train_resto, df_particion_train = umb.internal_train_partition(
    predicciones_originales['TRAIN'], CONFIG)
PARTITION_PATH = EVAL_ROOT / 'evidencias/particion_interna_train.csv'
df_particion_train.to_csv(PARTITION_PATH, index_label='_row_id')
display(df_particion_train.groupby('Particion').agg(N=('Target','size'), Defaults=('Target','sum'), Tasa_default=('Target','mean')))
print(umb.LIMITATION)
print('TRAIN_RESTO no se utiliza para elegir candidatos. No se entrena ningún modelo.')

## UMB.5 — Grilla en validación interna de TRAIN

**Recibe:** Solo train_validacion_interna y umbrales 0.01–0.50.

**Produce:** df_grilla_train_validacion.

**Funciones:** 5.1.0.evaluacion_umbrales.py: evaluate_grid, threshold_metrics.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: evaluate_grid, threshold_metrics
df_grilla_train_validacion = umb.evaluate_grid(train_validacion_interna)
display(df_grilla_train_validacion)
print('Precision/Recall/F1/Accuracy, TN/FP/FN/TP y volumen alto riesgo; NaN conserva causas explícitas.')

## UMB.6 — Candidatos TRAIN y congelación

**Recibe:** Grilla interna TRAIN y escenarios configurados.

**Produce:** df_umbrales_candidatos y JSON de umbrales congelados.

**Funciones:** 5.1.0.evaluacion_umbrales.py: choose_candidates, freeze_candidates.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: choose_candidates, freeze_candidates
df_umbrales_candidatos = umb.choose_candidates(df_grilla_train_validacion, CONFIG)
CANDIDATES_PATH = EVAL_ROOT / 'umbrales_candidatos_congelados.json'
artefacto_umbrales = umb.freeze_candidates(CANDIDATES_PATH, df_umbrales_candidatos, CONFIG,
    congelado['references'], PARTITION_PATH)
hash_umbrales_congelados = umb.sha256(CANDIDATES_PATH)
display(df_umbrales_candidatos)
df_candidatos_validacion_interna = umb.evaluate_candidates(
    {'TRAIN_VALIDACION_INTERNA': train_validacion_interna}, df_umbrales_candidatos)
display(df_candidatos_validacion_interna)
print('Escenarios diagnósticos. No se declara óptimo sin objetivo/costos FP/FN de negocio.')

## UMB.7 — Evaluación diagnóstica de candidatos congelados

**Recibe:** TRAIN/TEST/OOT y JSON ya persistido; no se eligen nuevos umbrales.

**Produce:** df_comparacion_umbrales.

**Funciones:** 5.1.0.evaluacion_umbrales.py: load_candidates, evaluate_candidates.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: load_candidates, evaluate_candidates
artefacto_recuperado, candidatos_recuperados = umb.load_candidates(CANDIDATES_PATH)
df_comparacion_umbrales = umb.evaluate_candidates(predicciones_originales, candidatos_recuperados)
display(df_comparacion_umbrales)
assert umb.sha256(CANDIDATES_PATH) == hash_umbrales_congelados
print('TEST/OOT: solo diagnóstico de los umbrales congelados en UMB.6.')

## UMB.8 — Grillas completas por muestra

**Recibe:** Predicciones originales; candidatos ya congelados.

**Produce:** df_grilla_umbrales y curvas Precision/Recall/F1.

**Funciones:** 5.1.0.evaluacion_umbrales.py: evaluate_grid, plot_thresholds.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: evaluate_grid, plot_thresholds
df_grilla_umbrales = pd.concat([
    umb.evaluate_grid(predicciones_originales[s]).assign(Segmento=s) for s in umb.SEGMENTS
], ignore_index=True)
display(df_grilla_umbrales)
figuras_umbrales['precision_recall_vs_umbral'] = umb.plot_thresholds(df_grilla_umbrales, candidatos_recuperados)
plt.show()
print('Las curvas TEST/OOT son descriptivas: no se usan para modificar candidatos.')

## UMB.9 — Captura acumulada y lift por deciles

**Recibe:** Predicciones congeladas por muestra.

**Produce:** df_deciles_riesgo, asignación de deciles y curva de captura.

**Funciones:** 5.1.0.evaluacion_umbrales.py: risk_deciles, plot_capture.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: risk_deciles, plot_capture
df_deciles_riesgo, df_asignacion_deciles = umb.risk_deciles(predicciones_originales)
display(df_deciles_riesgo)
print('Decil 1 = mayor riesgo. Deciles de tamaño casi igual por muestra; empates se ordenan por _row_id, sin target.')
print('Empates en frontera se registran: separar un score idéntico entre deciles no crea diferencias reales de riesgo ni un umbral operativo.')
figuras_umbrales['captura_acumulada'] = umb.plot_capture(df_deciles_riesgo)
plt.show()

## UMB.10 — Tasa observada y lift de cada decil

**Recibe:** df_deciles_riesgo.

**Produce:** Curva de default observado por decil.

**Funciones:** 5.1.0.evaluacion_umbrales.py: plot_decile_default.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: plot_decile_default
figuras_umbrales['default_por_decil'] = umb.plot_decile_default(df_deciles_riesgo)
plt.show()

In [ ]:
# UMB.10B — Lift por decil. Recibe df_deciles_riesgo; produce figura.
# Función: 5.1.0.evaluacion_umbrales.py: plot_lift
figuras_umbrales['lift_por_decil'] = umb.plot_lift(df_deciles_riesgo)
plt.show()

## UMB.11 — Calibración de probabilidades y Brier

**Recibe:** Predicciones originales, bins fijos [0,1].

**Produce:** df_calibracion_bins, df_calibracion_resumen y figura.

**Funciones:** 5.1.0.evaluacion_umbrales.py: calibration_diagnostics, plot_calibration.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: calibration_diagnostics, plot_calibration
df_calibracion_bins, df_calibracion_resumen = umb.calibration_diagnostics(predicciones_originales)
display(df_calibracion_resumen)
display(df_calibracion_bins)
figuras_umbrales['calibracion_probabilidades'] = umb.plot_calibration(df_calibracion_bins)
plt.show()
print('Bins vacíos: N=0 y tasas/promedios NaN. ECE depende del binning y es descriptivo.')
print('Brier independiente del umbral. Brier skill usa como referencia la prevalencia TRAIN constante.')
print('No se ajusta Platt, isotónica ni ningún otro calibrador.')

## UMB.12 — Validaciones y exportación

**Recibe:** Tablas y figuras en memoria; fuentes intactas.

**Produce:** Reportes, gráficos y evidencia de diagnóstico.

**Funciones:** 5.1.0.evaluacion_umbrales.py: export_diagnostics.

In [ ]:
# Funciones: 5.1.0.evaluacion_umbrales.py: export_diagnostics
pd.testing.assert_series_equal(modelo_congelado.params, coeficientes_iniciales, check_exact=True)
assert umb.sha256(CANDIDATES_PATH) == hash_umbrales_congelados
for path, digest in congelado['references'].items():
    assert umb.sha256(path) == digest, path
for s in umb.SEGMENTS:
    filas = df_grilla_umbrales.loc[df_grilla_umbrales.Segmento.eq(s)]
    assert filas[['TN','FP','FN','TP']].sum(axis=1).eq(len(predicciones_originales[s])).all()
    deciles = df_deciles_riesgo.loc[df_deciles_riesgo.Segmento.eq(s)]
    assert deciles.N.sum() == len(predicciones_originales[s])
    assert deciles.Defaults.sum() == predicciones_originales[s].Target.sum()
    assert np.isclose(deciles.iloc[-1].Captura_defaults_acumulada, 1.0)
umb.export_diagnostics(EVAL_ROOT, {
    'df_distribucion_probabilidades': df_distribucion_probabilidades,
    'df_grilla_train_validacion': df_grilla_train_validacion,
    'df_umbrales_candidatos': df_umbrales_candidatos,
    'df_candidatos_validacion_interna': df_candidatos_validacion_interna,
    'df_comparacion_umbrales': df_comparacion_umbrales,
    'df_grilla_umbrales': df_grilla_umbrales,
    'df_deciles_riesgo': df_deciles_riesgo,
    'df_asignacion_deciles': df_asignacion_deciles,
    'df_calibracion_bins': df_calibracion_bins,
    'df_calibracion_resumen': df_calibracion_resumen
}, figuras_umbrales)
congelado['validation'].to_csv(EVAL_ROOT / 'evidencias/validacion_fuentes.csv', index=False)
print('Exportado sin modificar modelo, coeficientes, WoE ni predicciones originales:', EVAL_ROOT)

## UMB.13 — Resultados y límites de interpretación

**Recibe:** Diagnóstico real y pruebas ejecutadas.

**Produce:** Resumen visible de candidatos y comparaciones.

**Funciones:** Resultados de 5.1.0.evaluacion_umbrales.py.

In [ ]:
# Funciones: Resultados de 5.1.0.evaluacion_umbrales.py
display(df_distribucion_probabilidades[['Segmento','Minimo','Promedio','P50','P90','P95','P99','Maximo','Pct_clientes_prob_ge_050']])
display(df_comparacion_umbrales[['Candidato','Segmento','Umbral','Precision','Recall','F1','Accuracy','Pct_clientes_alto_riesgo']])
display(df_deciles_riesgo.loc[df_deciles_riesgo.Decil.eq(1), ['Segmento','Pct_clientes','Tasa_default','Captura_defaults_acumulada','Lift_decil']])
display(df_calibracion_resumen)
print(umb.LIMITATION)
print('No hay umbral óptimo declarado: falta acordar objetivo/costos de negocio. Los candidatos no cambian el modelo 5.0.')
pruebas_path = EVAL_ROOT / 'evidencias/pruebas.json'
if pruebas_path.exists(): display(pd.DataFrame([json.loads(pruebas_path.read_text(encoding='utf-8'))]))